# Learn Spark

## Setting Up Spark

### Install PySpark

In [ ]:
!pip install pyspark

### Setting Up PySpark Session

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[*]").appName("Colab_PySpark_Test").getOrCreate()

### Importing Libraries

In [ ]:
from pyspark.sql.functions import *
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Sample Codes

#### Merge two Dataframes using PySpark


Here we use union() to merge two DataFrames. In there is a schema mismatch, then the table that is missing col.s are filled with null value using the function lit().

In [ ]:
simpleData1 = [(1,"Sagar","CSE","UP",80),\
    (2,'Shivam','IT','MP',86),\
        (3,'Priyanka','CSE','UP',85)]

columns = ["Id","Name","Dept","State","Score"]
df_1 = spark.createDataFrame(data = simpleData1, schema = columns)
df_1.show(5)

+---+--------+----+-----+-----+
| Id|    Name|Dept|State|Score|
+---+--------+----+-----+-----+
|  1|   Sagar| CSE|   UP|   80|
|  2|  Shivam|  IT|   MP|   86|
|  3|Priyanka| CSE|   UP|   85|
+---+--------+----+-----+-----+



In [ ]:
simpleData2 = [(4,'Priyanshu','IT','UP'),\
                (5,'Neha','CSE','UP')]

columns = ["Id","Name","Dept","State"]
df_2 = spark.createDataFrame(data = simpleData2, schema = columns)
df_2.show(5)

+---+---------+----+-----+
| Id|     Name|Dept|State|
+---+---------+----+-----+
|  4|Priyanshu|  IT|   UP|
|  5|     Neha| CSE|   UP|
+---+---------+----+-----+



In [ ]:
df_2 = df_2.withColumn("Score",lit(None))

In [ ]:
df_1.union(df_2).show()

+---+---------+----+-----+-----+
| Id|     Name|Dept|State|Score|
+---+---------+----+-----+-----+
|  1|    Sagar| CSE|   UP|   80|
|  2|   Shivam|  IT|   MP|   86|
|  3| Priyanka| CSE|   UP|   85|
|  4|Priyanshu|  IT|   UP| NULL|
|  5|     Neha| CSE|   UP| NULL|
+---+---------+----+-----+-----+



#### Explode columns using PySpark

explode() function is being used here which is used to transform an array i.e. list or map i.e. dict column into multiple rows. Each element in the array or map becomes a separate row in the resulting DataFrame.

Important Links:
<br>
- [https://rajanand.org/spark/spark-explode](url)
- [https://towardsdatascience.com/pyspark-explained-the-explode-and-collect-list-functions-834f45ff5ac5/](url)
- [https://bigdataenthusiast.medium.com/apache-spark-explode-function-f8c0ef87452](url)
- [https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.explode.html](url)

In [ ]:
# Exploding a list
data = [("Anand", ["Java", "Python"]),
        ("Bala", ["Scala", "Spark","Azure"]),
        ("Kavitha", ["SQL", "Hadoop"])]
columns = ["Name","Skills"]

df = spark.createDataFrame(data, columns)
df.show()

exploded_df = df.select("Name", explode("Skills").alias("Skills"))
exploded_df.show()

+-------+--------------------+
|   Name|              Skills|
+-------+--------------------+
|  Anand|      [Java, Python]|
|   Bala|[Scala, Spark, Az...|
|Kavitha|       [SQL, Hadoop]|
+-------+--------------------+

+-------+------+
|   Name|Skills|
+-------+------+
|  Anand|  Java|
|  Anand|Python|
|   Bala| Scala|
|   Bala| Spark|
|   Bala| Azure|
|Kavitha|   SQL|
|Kavitha|Hadoop|
+-------+------+



In [ ]:
# Exploding a Dictionary
data = [("James",{"Java":2,"Scala":2,"Python":4}),
        ("Michael",{"Spark":5,"Java":4}),
        ("Robert",{"CSharp":5}),
        ("Washington",{"C++":5})]

columns = ["name" ,"SkillLevel"]
df = spark.createDataFrame(data, columns)

exploded_df = df.select("name", explode("SkillLevel").alias("Skill","Level"))
exploded_df.show()

+----------+------+-----+
|      name| Skill|Level|
+----------+------+-----+
|     James|  Java|    2|
|     James| Scala|    2|
|     James|Python|    4|
|   Michael|  Java|    4|
|   Michael| Spark|    5|
|    Robert|CSharp|    5|
|Washington|   C++|    5|
+----------+------+-----+



#### Regex using PySpark

Here we are using rlike function which is similar to LIKE in SQL. It is used to filter DataFrame by matching a col.'s values against regular expression.

In [ ]:
sampleData1 = [
    (2, "Shivam", "U6754555iy"),
    (1, "Sagar", "2872587258"),
    (3, "Muni", "86468286U7")
]

columns = ["Id", "Name", "Mobile"]

df1 = spark.createDataFrame(sampleData1, columns)
display(df1)

DataFrame[Id: bigint, Name: string, Mobile: string]

In [ ]:
df1.select("*").filter(col("Mobile").rlike("^[0-9]+$")).show()

+---+-----+----------+
| Id| Name|    Mobile|
+---+-----+----------+
|  1|Sagar|2872587258|
+---+-----+----------+



#### Skip line while loading data into DataFrame

Here we will see methods to skip rows while loading data into PySpark DataFrame.
Mainly there are 3 methods:
1. Using skipRows
2. Using RDD
3. Using Pandas

In [ ]:
# Using skipRows
df1 = spark.read.option("header","true").option("skipRows",4).csv("/content/drive/MyDrive/Colab Notebooks/Spark/Data/SkipLineTestData.csv")

In [ ]:
df1.show()

+----+----+
| _c0| _c1|
+----+----+
|NULL|null|
|NULL|null|
|  AB|null|
|  CH|null|
|  ID|Name|
|   1|   A|
|   2|   B|
|   3|   C|
|   4|   D|
+----+----+



In [ ]:
# Using Pandas
dfpd = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Spark/Data/SkipLineTestData.csv")
df1 = spark.createDataFrame(dfpd[4:], schema =list(dfpd.columns))
df1.show()

+---+----+
|_c0| _c1|
+---+----+
| ID|Name|
|  1|   A|
|  2|   B|
|  3|   C|
|  4|   D|
+---+----+



In [ ]:
# Using RDD
rdd = spark.sparkContext.textFile("/content/drive/MyDrive/Colab Notebooks/Spark/Data/SkipLineTestData.csv")
rdd_skipped = rdd.zipWithIndex()
rdd_skipped.collect()

[('_c0,_c1', 0),
 ('NULL,null', 1),
 ('NULL,null', 2),
 ('AB,null', 3),
 ('CH,null', 4),
 ('ID,Name', 5),
 ('1,A', 6),
 ('2,B', 7),
 ('3,C', 8),
 ('4,D', 9)]

In [ ]:
rdd_skipped = rdd_skipped.filter(lambda x:x[1]>4)
df1 = spark.read.option("header","true").csv(rdd_skipped)
df1.show()

+----+-----+---+
|('ID|Name'| 5)|
+----+-----+---+
| ('1|   A'| 6)|
| ('2|   B'| 7)|
| ('3|   C'| 8)|
| ('4|   D'| 9)|
+----+-----+---+



#### Count Nulls in each column

Here, we are calculating the null count in each column of a DataFrame.
Here we are using isNull to check for Null values and to count Null values we are using when which returns 1 only when condition is true.

In [ ]:
df1 = spark.read.option("header",True).csv("/content/drive/MyDrive/Colab Notebooks/Spark/Data/RowCountTestData.csv")
df1.show()

+---+----+----+
| ID|Name| Age|
+---+----+----+
|  1|   A|  23|
|  2|   B|NULL|
|  3|   C|  56|
|  4|NULL|NULL|
|  5|NULL|NULL|
+---+----+----+



In [ ]:
df1.select([count(when(col(i).isNull(),1)) for i in df1.columns]).show()

+----------------------------------------+------------------------------------------+-----------------------------------------+
|count(CASE WHEN (ID IS NULL) THEN 1 END)|count(CASE WHEN (Name IS NULL) THEN 1 END)|count(CASE WHEN (Age IS NULL) THEN 1 END)|
+----------------------------------------+------------------------------------------+-----------------------------------------+
|                                       0|                                         2|                                        3|
+----------------------------------------+------------------------------------------+-----------------------------------------+



#### Handle multi delimiters

Here split() function is being used to handle multi delimited data. It takes 2 arguments, data and separator, and returns delimited data in list format.

In [ ]:
df1 = spark.read.option("header",True).csv("/content/drive/MyDrive/Colab Notebooks/Spark/Data/Multi-Delimiter.csv")
df1.show()

+---+----+---+--------+
| ID|Name|Age|   Marks|
+---+----+---+--------+
|  1|   A| 23|12|56|85|
|  2|   B| 24|35|36|87|
|  3|   C| 25|78|79|21|
|  4|   D| 26|65|73|87|
|  5|   E| 27|36|65|89|
+---+----+---+--------+



In [ ]:
df1 = df1.withColumn('Physics', split(col('Marks'),'\\|')[0])\
.withColumn('Maths',split(col('Marks'),'\\|')[1])\
.withColumn('Chemistry',split(col('Marks'),'\\|')[2])\
.drop('Marks')
df1.show()

+---+----+---+-------+-----+---------+
| ID|Name|Age|Physics|Maths|Chemistry|
+---+----+---+-------+-----+---------+
|  1|   A| 23|     12|   56|       85|
|  2|   B| 24|     35|   36|       87|
|  3|   C| 25|     78|   79|       21|
|  4|   D| 26|     65|   73|       87|
|  5|   E| 27|     36|   65|       89|
+---+----+---+-------+-----+---------+



#### Regex Replace

In [ ]:
df = spark.read.text('/content/drive/MyDrive/Colab Notebooks/Spark/Data/Regex_Replace.txt')
df.show()

+--------------------+
|               value|
+--------------------+
|1-A-12-2-B-23-3-C...|
+--------------------+



In [ ]:
pattern = rf"(.{{{7-1}}})."  # Generates: r"(.{2})."
replacement = r"$1,"          # Keeps first 2 chars, replaces 3rd with *
df1 = df.withColumn('values',regexp_replace(col('value'),pattern,replacement)).drop('value')
df1 = df1.withColumn('values',explode(split(col("values"),',')))
df1 = df1.withColumn('ID',split(col('values'),'-')[0]).withColumn('Name',split(col('values'),'-')[1]).withColumn('Age',split(col('values'),'-')[2]).drop('values')
df1.show()

+---+----+---+
| ID|Name|Age|
+---+----+---+
|  1|   A| 12|
|  2|   B| 23|
|  3|   C| 34|
|  4|   D| 15|
+---+----+---+



In [ ]:
df1 = df.withColumn('values',regexp_replace(col('value'),'(.*?\\-){3}','$0,')).drop('value')
df1 = df1.withColumn('values',explode(split(col("values"),'-,')))
df1 = df1.withColumn('ID',split(col('values'),'-')[0]).withColumn('Name',split(col('values'),'-')[1]).withColumn('Age',split(col('values'),'-')[2]).drop('values')
df1.show()

+---+----+---+
| ID|Name|Age|
+---+----+---+
|  1|   A| 12|
|  2|   B| 23|
|  3|   C| 34|
|  4|   D| 15|
+---+----+---+



#### Pivot and Explode

Here Pivot and Explode is being used to Pivot a particular dataset on Pivot column.
<br>
Pivot = Row -> Column
<br>
Unpivot = Column -> Row

To pivot a DataFrame on particular col.:
<br>
df.groupBy(<Unpivot Cols>).pivot(<Pivot Cols>).agg(collect_list(<Value Cols>))
<br>
Here, arrays_zip() is used to collect all the pivot cols so that explode can be applied on a single col.


In [ ]:
df1 = spark.read.format('csv').option('header',True).load('/content/drive/MyDrive/Colab Notebooks/Spark/Data/Pivot_Dataset.csv')
df1.show()

+---+----+-------+----------+
| ID|NAME|COUNTRY| Date_part|
+---+----+-------+----------+
|  1|Gaga|  India| 1/11/2022|
|  1|Katy|     UK| 1/11/2022|
|  1| Bey| Europe| 1/11/2022|
|  1|Gaga|     US| 1/11/2022|
|  2|Gaga|   NULL|10/11/2022|
|  2|Katy|  India|10/11/2022|
|  2| Bey|     US| 2/15/2022|
+---+----+-------+----------+



In [ ]:
df2 = df1.groupBy('ID', 'Date_part').pivot('Name').agg(collect_list('COUNTRY')).orderBy('ID')
df2 = df2.withColumn('new', arrays_zip("Bey","Gaga","Katy"))
df2 = df2.withColumn('new1', explode("new")).drop("new")
df2 = df2.select("ID","Date_part",'new1.Bey','new1.Gaga','new1.Katy')

In [ ]:
df2.show()

+---+----------+------+-----+-----+
| ID| Date_part|   Bey| Gaga| Katy|
+---+----------+------+-----+-----+
|  1| 1/11/2022|Europe|India|   UK|
|  1| 1/11/2022|  NULL|   US| NULL|
|  2| 2/15/2022|    US| NULL| NULL|
|  2|10/11/2022|  NULL| NULL|India|
+---+----------+------+-----+-----+



In [ ]:
df3 = df2.unpivot(
    ids=['ID','Date_part'],
    values=["Bey", "Gaga", "Katy"],
    variableColumnName="Name",
    valueColumnName="Country"
)

In [ ]:
df3.show()#filter(col('Country').isNotNull()).show()

+---+----------+----+-------+
| ID| Date_part|Name|Country|
+---+----------+----+-------+
|  1| 1/11/2022| Bey| Europe|
|  1| 1/11/2022|Gaga|  India|
|  1| 1/11/2022|Katy|     UK|
|  1| 1/11/2022| Bey|   NULL|
|  1| 1/11/2022|Gaga|     US|
|  1| 1/11/2022|Katy|   NULL|
|  2| 2/15/2022| Bey|     US|
|  2| 2/15/2022|Gaga|   NULL|
|  2| 2/15/2022|Katy|   NULL|
|  2|10/11/2022| Bey|   NULL|
|  2|10/11/2022|Gaga|   NULL|
|  2|10/11/2022|Katy|  India|
+---+----------+----+-------+



#### CollectSet

Here CollectSet is being used to group unique list of addresses into list for each custId.

In [ ]:
dataset = [{'custId':1, 'custName': 'Mark Ray', 'address':'AB'},
           {'custId':2, 'custName': 'Peter Smith', 'address':'CD'},
           {'custId':1, 'custName': 'Mark Ray', 'address':'EF'},
           {'custId':2, 'custName': 'Peter Smith', 'address':'GH'},
           {'custId':2, 'custName': 'Peter Smith', 'address':'CD'},
           {'custId':3, 'custName': 'Kate', 'address':'ID'}]


from pyspark.sql.types import *
schema = StructType([
    StructField('custId', IntegerType(), True),
    StructField('custName', StringType()),
    StructField('address', StringType())
])

df1 = spark.createDataFrame(data=dataset, schema=schema)
df1.show()

+------+-----------+-------+
|custId|   custName|address|
+------+-----------+-------+
|     1|   Mark Ray|     AB|
|     2|Peter Smith|     CD|
|     1|   Mark Ray|     EF|
|     2|Peter Smith|     GH|
|     2|Peter Smith|     CD|
|     3|       Kate|     ID|
+------+-----------+-------+



In [ ]:
df1.groupBy('custID', 'custName').agg(collect_set('address').alias('address')).orderBy('custID').show()

+------+-----------+--------+
|custID|   custName| address|
+------+-----------+--------+
|     1|   Mark Ray|[EF, AB]|
|     2|Peter Smith|[CD, GH]|
|     3|       Kate|    [ID]|
+------+-----------+--------+



#### Regexp_extract

Here regexp_extract function is being used to extract specific substrings from a column.  

In [ ]:
data=[('ABSHFJFJ12QWERT12',1),('QWERT5674OTUT1',2),('DGDGNJDJ1234UYI',3)]
df=spark.createDataFrame(data,schema="input_string string,id int")
df.show()

+-----------------+---+
|     input_string| id|
+-----------------+---+
|ABSHFJFJ12QWERT12|  1|
|   QWERT5674OTUT1|  2|
|  DGDGNJDJ1234UYI|  3|
+-----------------+---+



In [ ]:
df.select("*").\
withColumn("new_col1",regexp_extract(col("input_string"),'(^[a-zA-Z]*[0-9]*)',1)).\
withColumn("new_col2",regexp_extract(col("input_string"),'([a-zA-Z]*[0-9]*$)',1)).\
show()

+-----------------+---+------------+--------+
|     input_string| id|    new_col1|new_col2|
+-----------------+---+------------+--------+
|ABSHFJFJ12QWERT12|  1|  ABSHFJFJ12| QWERT12|
|   QWERT5674OTUT1|  2|   QWERT5674|   OTUT1|
|  DGDGNJDJ1234UYI|  3|DGDGNJDJ1234|     UYI|
+-----------------+---+------------+--------+



#### Next Question